# Intro to Machine Learning for Plant Sciences (ML4PS2026) - Deep Learning lab

## Introduction

Python has multiple high quality libraries for Deep Learning: **pytorch** (used here), **tensorflow**, **keras**. 

These libraries are standard, documented, fairly easy to use and highly optimized. They cover all relevant components for building custom training pipelines: data processing and loading, model design and training, optimization and parallelization, testing and evaluating.

Used in this lab:
- **torch** / **torchvision** for datasets, image transforms, models and training
- **segmentation_models_pytorch** for ready-made segmentation models (U-Net)
- **sklearn** for metrics
- **matplotlib** for plots

Other libraries that may be useful:
- **numpy** / **pandas** for handling data and datasets
- **rasterio** for handling raster data (eg satellite imagery)
- **wandb** (Weight and Biases) for hyperparameter tuning and keeping track of models trained
- **lightning** (on top of **pytorch**) for scalability and deployment with **pytorch**
... and so many more!

Outline of this lab:
1. Training a small CNN from scratch to classify images
2. Using a pretrained model (transfer learning)
3. Image segmentation


In [ ]:
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader, random_split
import torch.nn.functional as F
from torchvision import models
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet

from sklearn.metrics import accuracy_score

torch.manual_seed(42) # For reproducibility

# Use the GPU if one is available (much faster for training)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

### Training a Computer Vision model

Simple classification framework: we classify images of 37 cat and dog breeds (Oxford-IIIT Pet dataset).


#### Data

All images are brought to the same size (resized and center-cropped to 224x224 pixels, without distorting them) and normalized. We split the data into a **training** set (to learn the weights), a **validation** set (to compare models and settings) and a **test** set (held out to report the final performance).

In [ ]:
# Normalization
# Standard ImageNet normalization values (also expected by the pretrained models used below)
IMAGE_NET_MEAN = [0.485, 0.456, 0.406]
IMAGE_NET_STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize(224), # Scale the shorter side to 224 pixels (keeps the aspect ratio)
    transforms.CenterCrop(224), # Cut out the central 224x224 square
    transforms.ToTensor(),
    transforms.Normalize(IMAGE_NET_MEAN, IMAGE_NET_STD)
])

def denormalize(img):
    """Denormalize an ImageNet image tensor for visualization."""
    mean = torch.tensor(IMAGE_NET_MEAN).view(3,1,1)
    std = torch.tensor(IMAGE_NET_STD).view(3,1,1)
    img = img * std + mean 
    img = img.permute(1,2,0) 
    img = img.clamp(0,1)
    return img

In [ ]:
# Train Test Validation split

train_transform = transform # Add data augmentation here so it only affects the training set

# Load official datasets (trainval loaded twice so that train and val can use different transforms)
trainval_dataset = OxfordIIITPet(root='./data', split='trainval', download=True, transform=train_transform)
trainval_eval_dataset = OxfordIIITPet(root='./data', split='trainval', transform=transform)
test_dataset  = OxfordIIITPet(root='./data', split='test', download=True, transform=transform)
class_names = trainval_dataset.classes

# Create DataLoaders (same seed -> same split indices for both datasets)
train_dataset, _ = random_split(trainval_dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42))
_, val_dataset = random_split(trainval_eval_dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


In [ ]:
# Look at some training images
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(denormalize(img))
    ax.set_title(class_names[label])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Statistics
num_classes = len(class_names)
print(f"{len(train_dataset)} images (in train set) - {len(val_dataset)} images (in val set) - {len(test_dataset)} images (in test set) \n{num_classes} classes")

In [ ]:
# Check dataset balance
def plot_class_distribution(dataset, class_names):
    class_counts = [0] * len(class_names)
    for _, label in dataset:
        class_counts[label] += 1

    plt.figure(figsize=(10, 5))
    plt.bar(class_names, class_counts)
    plt.xticks(rotation=90)
    plt.xlabel('Classes')
    plt.ylabel('Number of images')
    plt.title('Class Distribution')
    plt.show() 

plot_class_distribution(train_dataset, class_names)

#### Model

In [ ]:
# Simple CNN model
class SimpleCNN(torch.nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        #self.conv2 = torch.nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = torch.nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc1 = torch.nn.Linear(16 * 112 * 112, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        #x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x
    
model = SimpleCNN(num_classes=len(class_names)) # Model instantiation

# Display the architecture of the model
print(model)

#### Training

Recap: a model is trained in **epochs** (one pass over the training data). In each step it predicts a **batch** of images, the **loss** measures how wrong the predictions are, and the **optimizer** updates the weights to reduce it. After each epoch we compute the loss on the validation set to monitor overfitting.

In [ ]:
# Training loop
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    """Train the model, print and plot train and validation loss per epoch"""
    model.to(device)
    train_losses, val_losses = list(), list()
    for epoch in range(num_epochs):
        model.train() # training mode
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)

        model.eval()  # evaluation mode
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
            val_loss /= len(val_loader)
            val_losses.append(val_loss)

        print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss:.4f} - Val Loss: {val_loss:.4f}')
    print('Training complete')

    # Plot training and validation loss
    plt.plot(range(num_epochs), train_losses, label='Train Loss')
    plt.plot(range(num_epochs), val_losses, label='Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()
    return train_losses, val_losses

In [ ]:
# Training 
criterion = torch.nn.CrossEntropyLoss() # Loss function
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9) # Optimizer
num_epochs = 10

train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs)

#### Evaluation

In [ ]:
# Evaluation function
def evaluate_model(model, loader):
    """Get predictions and true labels"""
    model.to(device)
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in loader:
            outputs = model(inputs.to(device))
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_labels, all_preds

In [ ]:
# Testing
all_labels, all_preds = evaluate_model(model, train_loader)
print(f"Accuracy on train set: {accuracy_score(all_labels, all_preds)}")

all_labels, all_preds = evaluate_model(model, test_loader)
print(f"Accuracy on test set: {accuracy_score(all_labels, all_preds)}")

In [ ]:
# TODO improve the model
# e.g. add the commented conv2 layer in SimpleCNN

# TODO change the learning rate, number of epochs, batch size

# TODO think about overfitting: regularization, dropout, data augmentation (via train_transform in the data cell), early stopping
# Possible methods can involve changes to the model, the optimizer or the data loading

# TODO add more metrics: confusion matrix, F1-score (see sklearn.metrics)

### More real use case: using a pretrained model

Recap: instead of starting from random weights, we start from a ResNet18 that was already trained on ImageNet (1.2M images, 1000 classes) and replace only its last layer to fit our 37 classes (**transfer learning**). The pretrained layers already detect useful visual features, so we need much less data and training time.

#### Data

We reuse the same datasets and loaders as above.

#### Model

In [ ]:
# Load pretrained ResNet18
resnet_model = models.resnet18(weights="ResNet18_Weights.IMAGENET1K_V1") # Model instantiation
resnet_model.fc = torch.nn.Linear(resnet_model.fc.in_features, num_classes) # Adapt last layer to current use case

# Display the architecture of the model
print(resnet_model)

#### Training

In [ ]:
# Training 
criterion = torch.nn.CrossEntropyLoss() # Loss function
optimizer = torch.optim.AdamW(resnet_model.parameters(), lr=1e-4, weight_decay=1e-4) # Optimizer
num_epochs = 10

train_losses, val_losses = train_model(resnet_model, train_loader, val_loader, criterion, optimizer, num_epochs)

In [ ]:
# TODO Try freezing all layers but the classifier head and retrain
if False:
    resnet_model = models.resnet18(weights="ResNet18_Weights.IMAGENET1K_V1") # Start again from the pretrained model
    for param in resnet_model.parameters():
        param.requires_grad = False
        
    resnet_model.fc = torch.nn.Linear(resnet_model.fc.in_features, num_classes) # New head, trainable by default
    optimizer = torch.optim.AdamW(resnet_model.fc.parameters(), lr=1e-4, weight_decay=1e-4) # Only optimize the head

#### Evaluation

In [ ]:
# TODO evaluate the ResNet model

#### So what is actually going on?

In [ ]:
# TODO check the output of the first layer
images, labels = next(iter(val_loader)) # One batch of validation images
# Single layers can be called like e.g. resnet_model.conv1
# What do the dimensions of the output correspond to?

In [ ]:
# TODO dive into class wise metrics
# e.g. precision and recall per class
# Which breeds are confused most often, and do they look alike?

### Doing image segmentation

Recap: in segmentation, the model predicts a label for **every pixel** instead of one label per image. Here we separate pet from background with a U-Net: an encoder (here a pretrained ResNet18) extracts features, and a decoder upsamples them back to a mask of the input size. With only these two classes, this is a binary rather than a multi-class segmentation task, and a binary cross-entropy loss (e.g. `BCEWithLogitsLoss`) is a good default to start with.

In [ ]:
import segmentation_models_pytorch as smp

#### Data

In [ ]:
# Transforms for images and masks
img_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGE_NET_MEAN, IMAGE_NET_STD)
])

# Masks (trimaps) have pixel values 1 = pet, 2 = background, 3 = border
mask_transform = transforms.Compose([
    transforms.Resize((128, 128), interpolation=transforms.InterpolationMode.NEAREST), # NEAREST keeps valid label values
    transforms.PILToTensor(), 
    transforms.Lambda(lambda mask: (mask == 1).float()), # Convert masks to 0/1 (background vs pet)
])

# Load official datasets (with masks)
trainval_seg_dataset = OxfordIIITPet(root='./data', split='trainval', download=True, transform=img_transform,
                                     target_types='segmentation', target_transform=mask_transform)
test_seg_dataset = OxfordIIITPet(root='./data', split='test', download=True, transform=img_transform,
                                 target_types='segmentation', target_transform=mask_transform)

# Create DataLoaders
train_seg_dataset, val_seg_dataset = random_split(trainval_seg_dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42))
train_seg_loader = DataLoader(train_seg_dataset, batch_size=4, shuffle=True)
val_seg_loader = DataLoader(val_seg_dataset, batch_size=4, shuffle=False)
test_seg_loader = DataLoader(test_seg_dataset, batch_size=4, shuffle=False)

In [ ]:
images, masks = next(iter(train_seg_loader)) 

masks.unique(return_counts=True) # Pixel counts per class (0 = background, 1 = pet)

#### Model

In [ ]:
# Load pretrained U-Net (encoder pretrained on ImageNet)
model = smp.Unet(encoder_name="resnet18", encoder_weights="imagenet", in_channels=3, classes=1).to(device)

#### Training

Try doing it all yourself on this image segmentation task. Only the encoder is pretrained, the decoder starts from random weights, so before training the predicted masks are essentially random (you can run the evaluation cells below first to see this).

In [ ]:
# TODO train the U-Net
# The train_model function from above can be reused here
# How do the predictions below compare to the ones before training?

#### Evaluation

In [ ]:
# Predict on one validation batch
model.eval()

images, masks = next(iter(val_seg_loader)) 
with torch.no_grad():
    outputs = model(images.to(device))
    preds = torch.sigmoid(outputs).cpu()

In [ ]:
# Visualize the predictions
fig, axes = plt.subplots(4, 3, figsize=(12, 12))
for i in range(4):
    axes[i,0].imshow(denormalize(images[i]))
    axes[i,0].set_title("Image")
    axes[i,0].axis('off')
    
    axes[i,1].imshow(masks[i][0], cmap='gray')
    axes[i,1].set_title("True Mask")
    axes[i,1].axis('off')
    
    axes[i,2].imshow(preds[i][0], cmap='gray')
    axes[i,2].set_title("Pred Mask")
    axes[i,2].axis('off')

plt.tight_layout()
plt.show()